# Cell 1 — Setup

In [1]:
from google.colab import drive
import os
drive.mount('/content/drive')

!pip install stable-baselines3[extra] gymnasium pygame -q
!apt-get install -y ffmpeg -q
print("✓ Setup complete")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 15.9 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
✓ Setup complete


# Cell 2 — Environment (fixed, standalone)

In [2]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import pygame

class AdversarialEnv(gym.Env):

    def __init__(self, render_mode=None):
        super().__init__()
        self.bg_max_speed  = 0.08
        self.int_max_speed = 0.05
        self.arena_size    = 1.0
        self.zone_center   = np.array([0.15, 0.85], dtype=np.float32)
        self.zone_bounds   = {'x': (0.0, 0.3), 'y': (0.7, 1.0)}
        self.intercept_radius = 0.10
        self.max_steps     = 400
        self.R_BLOCK_PER_STEP  =  0.30
        self.R_CLOSE_SCALE     =  2.0
        self.R_INTERCEPT       = 50.0
        self.R_ZONE_PENALTY    = -5.0
        self.R_BREACH_TERMINAL = -30.0
        self.R_TIMEOUT_WIN     = 20.0

        self.I_APPROACH_SCALE  =  2.0
        self.I_IN_ZONE         = 30.0
        self.I_INTERCEPTED     = -30.0
        self.I_TIMEOUT_LOSS    = -10.0
        self.observation_space = spaces.Box(-2.0, 2.0, shape=(14,), dtype=np.float32)
        self.action_space      = spaces.Box(-1.0, 1.0, shape=(4,), dtype=np.float32)
        self.render_mode = render_mode
        self.screen = None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        rng = self.np_random
        self.int_pos = np.array([
            rng.uniform(0.55, 0.92),
            rng.uniform(0.05, 0.50)
        ], dtype=np.float32)
        mid = (self.int_pos + self.zone_center) * 0.5
        noise = rng.uniform(-0.10, 0.10, size=2).astype(np.float32)
        self.bg_pos = np.clip(mid + noise, 0.05, 0.95)

        self.bg_vel  = np.zeros(2, dtype=np.float32)
        self.int_vel = np.zeros(2, dtype=np.float32)
        self.step_num = 0
        self.prev_int_to_zone = self._dist(self.int_pos, self.zone_center)
        self.prev_bg_to_int   = self._dist(self.bg_pos, self.int_pos)
        return self._obs(), {}

    def _obs(self):
        v  = self.zone_center - self.int_pos
        d  = np.linalg.norm(v) + 1e-8
        uv = v / d
        bs = self._blocking_score()
        return np.array([
            self.bg_pos[0],  self.bg_pos[1],
            self.bg_vel[0],  self.bg_vel[1],
            self.int_pos[0], self.int_pos[1],
            self.int_vel[0], self.int_vel[1],
            self.zone_center[0], self.zone_center[1],
            uv[0], uv[1],
            bs,
            self.step_num / self.max_steps,
        ], dtype=np.float32)

    def _move(self, pos, vel, raw, max_speed):
        vel = vel * 0.80 + raw.astype(np.float32) * 0.018
        spd = np.linalg.norm(vel)
        if spd > max_speed:
            vel = vel / spd * max_speed
        return np.clip(pos + vel, 0.0, 1.0), vel

    def _dist(self, a, b):
        return float(np.linalg.norm(a - b))

    def _in_zone(self, pos):
        return (self.zone_bounds['x'][0] <= pos[0] <= self.zone_bounds['x'][1] and
                self.zone_bounds['y'][0] <= pos[1] <= self.zone_bounds['y'][1])

    def _blocking_score(self):
        """Returns 0..1: how well is BG on the intruder→zone line?"""
        v   = self.zone_center - self.int_pos
        d2  = float(np.dot(v, v)) + 1e-8
        t   = float(np.dot(self.bg_pos - self.int_pos, v)) / d2
        t   = np.clip(t, 0.0, 1.0)
        proj = self.int_pos + t * v
        off  = self._dist(self.bg_pos, proj)
        return float(np.exp(-off * 10.0))

    def step(self, action):
        self.step_num += 1

        self.bg_pos,  self.bg_vel  = self._move(self.bg_pos,  self.bg_vel,  action[0:2], self.bg_max_speed)
        self.int_pos, self.int_vel = self._move(self.int_pos, self.int_vel, action[2:4], self.int_max_speed)

        bg_to_int   = self._dist(self.bg_pos, self.int_pos)
        int_to_zone = self._dist(self.int_pos, self.zone_center)
        blocking    = self._blocking_score()

        intercepted   = bg_to_int   < self.intercept_radius
        zone_breached = self._in_zone(self.int_pos)
        bg_r  = self.R_BLOCK_PER_STEP * blocking
        bg_r += self.R_CLOSE_SCALE * max(0.0, self.prev_bg_to_int - bg_to_int)
        if int_to_zone < 0.30:
            bg_r += self.R_ZONE_PENALTY * (0.30 - int_to_zone) / 0.30
        int_r  = self.I_APPROACH_SCALE * max(0.0, self.prev_int_to_zone - int_to_zone)

        terminated = truncated = False
        success = False

        if intercepted:
            bg_r  += self.R_INTERCEPT
            int_r += self.I_INTERCEPTED
            terminated = True
            success    = True
        elif zone_breached:
            bg_r  += self.R_BREACH_TERMINAL
            int_r += self.I_IN_ZONE
            terminated = True
        elif self.step_num >= self.max_steps:
            bg_r  += self.R_TIMEOUT_WIN
            int_r += self.I_TIMEOUT_LOSS
            truncated = True
            success   = True

        self.prev_bg_to_int   = bg_to_int
        self.prev_int_to_zone = int_to_zone

        return self._obs(), bg_r, int_r, terminated, truncated, {
            'intercepted':    intercepted,
            'zone_breached':  zone_breached,
            'success':        success,
            'blocking':       blocking,
            'int_to_zone':    int_to_zone,
            'step':           self.step_num,
        }

    def render(self):
        if self.render_mode != 'rgb_array':
            return None
        if self.screen is None:
            pygame.init()
            self.screen = pygame.Surface((500, 500))
        self.screen.fill((240, 240, 240))

        def to_px(pos):
            return int(pos[0] * 500), int((1 - pos[1]) * 500)


        zx = int(self.zone_bounds['x'][0] * 500)
        zy = int((1 - self.zone_bounds['y'][1]) * 500)
        zw = int((self.zone_bounds['x'][1] - self.zone_bounds['x'][0]) * 500)
        zh = int((self.zone_bounds['y'][1] - self.zone_bounds['y'][0]) * 500)
        pygame.draw.rect(self.screen, (255, 160, 160), (zx, zy, zw, zh))
        pygame.draw.rect(self.screen, (180, 40, 40),   (zx, zy, zw, zh), 2)
        bg_px = to_px(self.bg_pos)
        pygame.draw.circle(self.screen, (100, 160, 255), bg_px,
                           int(self.intercept_radius * 500), 1)

        pygame.draw.circle(self.screen, (20, 90, 210), bg_px, 13)
        pygame.draw.circle(self.screen, (210, 50, 20), to_px(self.int_pos), 10)

        return np.transpose(pygame.surfarray.pixels3d(self.screen), (1, 0, 2))

print("✅ AdversarialEnv ready")
print(f"   BG speed={0.08} | Intruder speed={0.05} | intercept_r={0.10}")

✅ AdversarialEnv ready
   BG speed=0.08 | Intruder speed=0.05 | intercept_r=0.1


# Cell 3 — Wrappers + Training (SAC for BG, PPO for Intruder)

In [13]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import pygame

class AdversarialEnv(gym.Env):
    def __init__(self, intruder_speed=0.03, render_mode=None):
        super().__init__()
        self.bg_max_speed    = 0.08
        self.int_max_speed   = intruder_speed
        self.arena_size      = 1.0
        self.zone_center     = np.array([0.15, 0.85], dtype=np.float32)
        self.zone_bounds     = {'x': (0.0, 0.3), 'y': (0.7, 1.0)}
        self.intercept_radius = 0.12
        self.max_steps       = 400

        self.R_BLOCK   =  0.40
        self.R_CLOSE   =  3.0
        self.R_WIN     = 60.0
        self.R_LOSE    = -25.0
        self.R_TIMEOUT = 25.0
        self.I_APPROACH =  3.0
        self.I_WIN      = 40.0
        self.I_LOSE     = -25.0
        self.I_TIMEOUT  = -15.0

        self.observation_space = spaces.Box(-2.0, 2.0, (14,), dtype=np.float32)
        self.action_space      = spaces.Box(-1.0,  1.0, (4,), dtype=np.float32)
        self.render_mode = render_mode
        self.screen = None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        rng = self.np_random
        self.int_pos = np.array([
            rng.uniform(0.60, 0.92),
            rng.uniform(0.05, 0.45)
        ], dtype=np.float32)
        mid   = (self.int_pos + self.zone_center) * 0.5
        noise = rng.uniform(-0.08, 0.08, 2).astype(np.float32)
        self.bg_pos  = np.clip(mid + noise, 0.05, 0.95)
        self.bg_vel  = np.zeros(2, dtype=np.float32)
        self.int_vel = np.zeros(2, dtype=np.float32)
        self.step_num   = 0
        self.prev_i2z   = self._dist(self.int_pos, self.zone_center)
        self.prev_b2i   = self._dist(self.bg_pos,  self.int_pos)
        return self._obs(), {}

    def _obs(self):
        v  = self.zone_center - self.int_pos
        d  = np.linalg.norm(v) + 1e-8
        bs = self._block()
        return np.array([
            self.bg_pos[0],  self.bg_pos[1],
            self.bg_vel[0],  self.bg_vel[1],
            self.int_pos[0], self.int_pos[1],
            self.int_vel[0], self.int_vel[1],
            self.zone_center[0], self.zone_center[1],
            v[0]/d, v[1]/d, bs,
            self.step_num / self.max_steps,
        ], dtype=np.float32)

    def _move(self, pos, vel, raw, spd):
        vel = vel * 0.78 + raw.astype(np.float32) * 0.020
        n = np.linalg.norm(vel)
        if n > spd: vel = vel / n * spd
        return np.clip(pos + vel, 0.0, 1.0), vel

    def _dist(self, a, b):
        return float(np.linalg.norm(a - b))

    def _in_zone(self, p):
        return (self.zone_bounds['x'][0] <= p[0] <= self.zone_bounds['x'][1] and
                self.zone_bounds['y'][0] <= p[1] <= self.zone_bounds['y'][1])

    def _block(self):
        v  = self.zone_center - self.int_pos
        d2 = float(np.dot(v, v)) + 1e-8
        t  = float(np.clip(np.dot(self.bg_pos - self.int_pos, v) / d2, 0, 1))
        return float(np.exp(-self._dist(self.bg_pos, self.int_pos + t*v) * 10.0))

    def step(self, action):
        self.step_num += 1
        self.bg_pos,  self.bg_vel  = self._move(self.bg_pos,  self.bg_vel,  action[0:2], self.bg_max_speed)
        self.int_pos, self.int_vel = self._move(self.int_pos, self.int_vel, action[2:4], self.int_max_speed)

        b2i = self._dist(self.bg_pos, self.int_pos)
        i2z = self._dist(self.int_pos, self.zone_center)
        blk = self._block()
        hit = b2i < self.intercept_radius
        brd = self._in_zone(self.int_pos)

        bg_r  = self.R_BLOCK * blk + self.R_CLOSE * max(0, self.prev_b2i - b2i)
        int_r = self.I_APPROACH * max(0, self.prev_i2z - i2z)

        term = trunc = False
        success = False
        if hit:
            bg_r += self.R_WIN;     int_r += self.I_LOSE
            term = True;            success = True
        elif brd:
            bg_r += self.R_LOSE;    int_r += self.I_WIN
            term = True
        elif self.step_num >= self.max_steps:
            bg_r += self.R_TIMEOUT; int_r += self.I_TIMEOUT
            trunc = True;           success = True

        self.prev_b2i = b2i
        self.prev_i2z = i2z
        return self._obs(), bg_r, int_r, term, trunc, {
            'success': success, 'intercepted': hit,
            'zone_breached': brd, 'blocking': blk,
            'int_to_zone': i2z, 'step': self.step_num,
        }

    def render(self):
        if self.render_mode != 'rgb_array': return None
        if self.screen is None:
            pygame.init()
            self.screen = pygame.Surface((500, 500))
        self.screen.fill((240, 240, 240))
        def px(p): return int(p[0]*500), int((1-p[1])*500)
        zx = int(self.zone_bounds['x'][0]*500)
        zy = int((1-self.zone_bounds['y'][1])*500)
        zw = int((self.zone_bounds['x'][1]-self.zone_bounds['x'][0])*500)
        zh = int((self.zone_bounds['y'][1]-self.zone_bounds['y'][0])*500)
        pygame.draw.rect(self.screen, (255,160,160), (zx,zy,zw,zh))
        pygame.draw.rect(self.screen, (180,40,40),   (zx,zy,zw,zh), 2)
        bp = px(self.bg_pos); ip = px(self.int_pos)
        pygame.draw.circle(self.screen, (100,160,255), bp, int(self.intercept_radius*500), 1)
        pygame.draw.circle(self.screen, (20,90,210),   bp, 13)
        pygame.draw.circle(self.screen, (210,50,20),   ip, 10)
        return np.transpose(pygame.surfarray.pixels3d(self.screen), (1,0,2))

print("✅ AdversarialEnv ready")
print(f"   BG speed=0.08 (fixed) | Intruder speed=variable | intercept_r=0.12")

✅ AdversarialEnv ready
   BG speed=0.08 (fixed) | Intruder speed=variable | intercept_r=0.12


In [14]:

from stable_baselines3 import SAC, PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback

MODEL = '/content/models'

class BGEnv(gym.Env):
    def __init__(self, int_policy=None, int_speed=0.03, scripted_prob=1.0):
        super().__init__()
        self.env           = AdversarialEnv(intruder_speed=int_speed)
        self.int_policy    = int_policy
        self.scripted_prob = scripted_prob
        self.observation_space = self.env.observation_space
        self.action_space  = spaces.Box(-1., 1., (2,), dtype=np.float32)

    def reset(self, seed=None, options=None):
        return self.env.reset(seed=seed, options=options)

    def step(self, bg_a):
        obs = self.env._obs()
        if self.int_policy is None or np.random.random() < self.scripted_prob:
            d     = self.env.zone_center - self.env.int_pos
            int_a = (d / (np.linalg.norm(d) + 1e-8)).astype(np.float32)
        else:
            int_a, _ = self.int_policy.predict(obs, deterministic=False)
        obs, bg_r, _, term, trunc, info = self.env.step(np.concatenate([bg_a, int_a]))
        return obs, bg_r, term, trunc, info


class INTEnv(gym.Env):
    def __init__(self, bg_policy, int_speed=0.03):
        super().__init__()
        self.env       = AdversarialEnv(intruder_speed=int_speed)
        self.bg_policy = bg_policy
        self.observation_space = self.env.observation_space
        self.action_space  = spaces.Box(-1., 1., (2,), dtype=np.float32)

    def reset(self, seed=None, options=None):
        return self.env.reset(seed=seed, options=options)

    def step(self, int_a):
        obs = self.env._obs()
        bg_a, _ = self.bg_policy.predict(obs, deterministic=True)
        obs, _, int_r, term, trunc, info = self.env.step(np.concatenate([bg_a, int_a]))
        return obs, int_r, term, trunc, info

class LogCB(BaseCallback):
    def __init__(self, freq=20_000, label=""):
        super().__init__()
        self.freq  = freq
        self.label = label
        self.t0    = time.time()

    def _on_step(self):
        if self.num_timesteps % self.freq == 0:
            mins = (time.time() - self.t0) / 60
            print(f"  [{self.label}] {self.num_timesteps:,} steps | {mins:.1f} min", flush=True)
        return True

def evaluate(bg, int_p, n=30, int_speed=0.03, label=""):
    wins = intercepts = breaches = 0
    for _ in range(n):
        env = AdversarialEnv(intruder_speed=int_speed)
        obs, _ = env.reset()
        done = False
        while not done:
            ba, _ = bg.predict(obs,    deterministic=True)
            ia, _ = int_p.predict(obs, deterministic=True)
            obs, _, _, term, trunc, info = env.step(np.concatenate([ba, ia]))
            done = term or trunc
        wins       += int(info['success'])
        intercepts += int(info['intercepted'])
        breaches   += int(info['zone_breached'])
    rate = wins / n * 100
    print(f"  {label} | {wins}/{n} wins ({rate:.0f}%) "
          f"intercepts={intercepts} breaches={breaches}", flush=True)
    return rate
def train_bg_until(bg_model, int_policy, int_speed,
                   scripted_prob, target_rate,
                   max_steps, label, chunk=30_000):
    trained = 0
    best    = 0.0
    while trained < max_steps:
        bg_model.learn(chunk, reset_num_timesteps=False,
                       callback=LogCB(chunk, label))
        trained += chunk
        wins = 0
        for _ in range(20):
            env = AdversarialEnv(intruder_speed=int_speed)
            obs, _ = env.reset(); done = False
            while not done:
                ba, _ = bg_model.predict(obs, deterministic=True)
                if int_policy is None:
                    d  = env.zone_center - env.int_pos
                    ia = (d / (np.linalg.norm(d)+1e-8)).astype(np.float32)
                else:
                    ia, _ = int_policy.predict(obs, deterministic=True)
                obs, _, _, term, trunc, info = env.step(np.concatenate([ba, ia]))
                done = term or trunc
            wins += int(info['success'])
        rate = wins / 20 * 100
        best = max(best, rate)
        print(f"  [{label}] {trained:,} steps → {wins}/20 ({rate:.0f}%)", flush=True)

        if rate >= target_rate:
            print(f"  Target {target_rate}% hit! Moving on.", flush=True)
            return bg_model, rate, True

    print(f"  Max steps done. Best={best:.0f}%", flush=True)
    return bg_model, best, False
SAC_CFG = dict(
    policy='MlpPolicy',
    learning_rate=3e-4,
    buffer_size=200_000,
    learning_starts=2_000,
    batch_size=512,
    tau=0.005,
    gamma=0.99,
    train_freq=4,
    gradient_steps=4,
    ent_coef='auto',
    policy_kwargs=dict(net_arch=[256, 256, 128]),
    verbose=0,
    device='cpu',
)

PPO_CFG = dict(
    policy='MlpPolicy',
    learning_rate=2e-4,
    n_steps=2048,
    batch_size=256,
    n_epochs=8,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.02,
    policy_kwargs=dict(net_arch=[256, 256]),
    verbose=0,
    device='cpu',
)

print("✅ Wrappers, helpers and configs ready")

✅ Wrappers, helpers and configs ready


In [15]:
print("=" * 55)
print("STAGE 0 — Intruder speed=0.03, scripted (target 80%)")
print("=" * 55)

S0_SPEED = 0.03

bg_env   = BGEnv(int_policy=None, int_speed=S0_SPEED, scripted_prob=1.0)
bg_model = SAC(env=Monitor(bg_env), **SAC_CFG)

bg_model, rate0, ok0 = train_bg_until(
    bg_model,
    int_policy    = None,
    int_speed     = S0_SPEED,
    scripted_prob = 1.0,
    target_rate   = 80.0,
    max_steps     = 150_000,
    label         = "BG-S0",
    chunk         = 30_000,
)
bg_policy = bg_model
bg_model.save(f'{MODEL}/bg_stage0')

print(f"\nStage 0 BG done — best rate={rate0:.0f}%")
if rate0 < 60:
    print("  Still low — consider re-running this cell before continuing")
else:
    print("  Good! Training intruder at stage-0 speed...")
int_env   = INTEnv(bg_policy=bg_policy, int_speed=S0_SPEED)
int_model = PPO(env=Monitor(int_env), **PPO_CFG)
int_model.learn(60_000, callback=LogCB(20_000, "INT-S0"))
int_policy = int_model
int_model.save(f'{MODEL}/int_stage0')

print("\nStage 0 final head-to-head:")
evaluate(bg_policy, int_policy, n=30, int_speed=S0_SPEED, label="[S0]")

STAGE 0 — Intruder speed=0.03, scripted (target 80%)
  [BG-S0] 30,000 steps | 25.7 min
  [BG-S0] 30,000 steps → 20/20 (100%)
  Target 80.0% hit! Moving on.

Stage 0 BG done — best rate=100%
  Good! Training intruder at stage-0 speed...
  [INT-S0] 20,000 steps | 0.6 min
  [INT-S0] 40,000 steps | 1.2 min
  [INT-S0] 60,000 steps | 1.8 min

Stage 0 final head-to-head:
  [S0] | 30/30 wins (100%) intercepts=9 breaches=0


100.0

In [18]:
print("=" * 55)
print("STAGE 1 — Intruder speed=0.04, learned (target 60%)")
print("=" * 55)

S1_SPEED = 0.04

bg_env  = BGEnv(int_policy=int_policy, int_speed=S1_SPEED, scripted_prob=0.2)
new_bg  = SAC(env=Monitor(bg_env), **SAC_CFG)
new_bg.set_parameters(bg_policy.get_parameters())

new_bg, rate1, ok1 = train_bg_until(
    new_bg,
    int_policy    = int_policy,
    int_speed     = S1_SPEED,
    scripted_prob = 0.2,
    target_rate   = 60.0,
    max_steps     = 180_000,
    label         = "BG-S1",
    chunk         = 30_000,
)
bg_policy = new_bg
new_bg.save(f'{MODEL}/bg_stage1')

print(f"\nStage 1 BG done — best rate={rate1:.0f}%")

int_env   = INTEnv(bg_policy=bg_policy, int_speed=S1_SPEED)
new_int   = PPO(env=Monitor(int_env), **PPO_CFG)
new_int.learn(80_000, callback=LogCB(20_000, "INT-S1"))
int_policy = new_int
new_int.save(f'{MODEL}/int_stage1')

print("\nStage 1 final head-to-head:")
evaluate(bg_policy, int_policy, n=30, int_speed=S1_SPEED, label="[S1]")

STAGE 1 — Intruder speed=0.04, learned (target 60%)
  [BG-S1] 30,000 steps | 27.7 min
  [BG-S1] 30,000 steps → 20/20 (100%)
  Target 60.0% hit! Moving on.

Stage 1 BG done — best rate=100%
  [INT-S1] 20,000 steps | 0.6 min
  [INT-S1] 40,000 steps | 1.2 min
  [INT-S1] 60,000 steps | 1.8 min
  [INT-S1] 80,000 steps | 2.4 min

Stage 1 final head-to-head:
  [S1] | 30/30 wins (100%) intercepts=0 breaches=0


100.0

STAGE 1 — Intruder speed=0.04, learned (target 60%)
  [BG-S1] 30,000 steps | 28.2 min
  [BG-S1] 30,000 steps → 20/20 (100%)
  Target 60.0% hit! Moving on.

Stage 1 BG done — best rate=100%
  [INT-S1] 20,000 steps | 0.6 min
  [INT-S1] 40,000 steps | 1.2 min
  [INT-S1] 60,000 steps | 1.8 min
  [INT-S1] 80,000 steps | 2.5 min

Stage 1 final head-to-head:
  [S1] | 0/30 wins (0%) intercepts=0 breaches=30


0.0

In [19]:
print("=" * 55)
print("STAGE 2 — Full speed intruder=0.05, self-play 3 rounds")
print("=" * 55)

S2_SPEED = 0.05
ROUNDS   = 3

for r in range(1, ROUNDS + 1):
    print(f"\n── Round {r}/{ROUNDS} ───────────────────────────────", flush=True)
    t0 = time.time()
    bg_env = BGEnv(int_policy=int_policy, int_speed=S2_SPEED, scripted_prob=0.0)
    new_bg = SAC(env=Monitor(bg_env), **SAC_CFG)
    new_bg.set_parameters(bg_policy.get_parameters())
    new_bg.learn(100_000, reset_num_timesteps=False,
                 callback=LogCB(25_000, f"BG-S2-r{r}"))
    bg_policy = new_bg
    new_bg.save(f'{MODEL}/bg_s2_r{r}')
    int_env = INTEnv(bg_policy=bg_policy, int_speed=S2_SPEED)
    new_int = PPO(env=Monitor(int_env), **PPO_CFG)
    new_int.learn(80_000, callback=LogCB(20_000, f"INT-S2-r{r}"))
    int_policy = new_int
    new_int.save(f'{MODEL}/int_s2_r{r}')

    mins = (time.time() - t0) / 60
    print(f"\nRound {r} eval ({mins:.1f} min):", flush=True)
    evaluate(bg_policy, int_policy, n=40, int_speed=S2_SPEED, label=f"[S2-r{r}]")
bg_policy.save(f'{MODEL}/bodyguard_final')
int_policy.save(f'{MODEL}/intruder_final')
print("\n✅ Training complete — finals saved to /content/models/")

STAGE 2 — Full speed intruder=0.05, self-play 3 rounds

── Round 1/3 ───────────────────────────────
  [BG-S2-r1] 25,000 steps | 23.6 min
  [BG-S2-r1] 50,000 steps | 49.4 min
  [BG-S2-r1] 75,000 steps | 75.5 min
  [BG-S2-r1] 100,000 steps | 102.0 min
  [INT-S2-r1] 20,000 steps | 0.6 min
  [INT-S2-r1] 40,000 steps | 1.2 min
  [INT-S2-r1] 60,000 steps | 1.8 min
  [INT-S2-r1] 80,000 steps | 2.4 min

Round 1 eval (104.5 min):
  [S2-r1] | 0/40 wins (0%) intercepts=0 breaches=40

── Round 2/3 ───────────────────────────────


KeyboardInterrupt: 

In [20]:
from stable_baselines3 import SAC, PPO
bg  = SAC.load('/content/models/bodyguard_final')
itr = PPO.load('/content/models/intruder_final')

wins = 0
print("50-episode evaluation\n" + "─" * 52)

for ep in range(50):
    env = AdversarialEnv(intruder_speed=0.05)
    obs, _ = env.reset()
    done = False
    steps = 0
    while not done:
        ba, _ = bg.predict(obs,  deterministic=True)
        ia, _ = itr.predict(obs, deterministic=True)
        obs, _, _, term, trunc, info = env.step(np.concatenate([ba, ia]))
        steps += 1
        done = term or trunc

    wins += int(info['success'])
    if info['success']:
        tag = f"WIN  ({'intercept' if info['intercepted'] else 'timeout '})"
    else:
        tag = "BREACH"
    print(f"Ep {ep+1:02d}: {tag} | steps={steps:3d} "
          f"block={info['blocking']:.2f} zone_d={info['int_to_zone']:.2f}")

print(f"\n{'='*52}")
print(f"FINAL RESULT: {wins}/50 = {wins*2}% bodyguard win rate")
print(f"{'='*52}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/models/bodyguard_final.zip'

In [21]:
import imageio
import pygame
from IPython.display import Image, display
from stable_baselines3 import SAC, PPO
MODEL_DIR = '/content/models'
GIF_PATH  = '/content/gifs/bodyguard.gif'

bg  = SAC.load(f'{MODEL_DIR}/bodyguard_final')
itr = PPO.load(f'{MODEL_DIR}/intruder_final')

EPISODES = 5
FPS      = 18
PX       = 500

pygame.init()

def draw_frame(env, step, ep, banner=""):
    surf  = pygame.Surface((PX, PX))
    surf.fill((240, 240, 240))
    font  = pygame.font.SysFont('monospace', 13)
    font2 = pygame.font.SysFont('monospace', 20, bold=True)

    def px(p):
        return int(p[0]*PX), int((1-p[1])*PX)
    zx = int(env.zone_bounds['x'][0]*PX)
    zy = int((1-env.zone_bounds['y'][1])*PX)
    zw = int((env.zone_bounds['x'][1]-env.zone_bounds['x'][0])*PX)
    zh = int((env.zone_bounds['y'][1]-env.zone_bounds['y'][0])*PX)
    pygame.draw.rect(surf, (255,160,160), (zx,zy,zw,zh))
    pygame.draw.rect(surf, (180,40,40),   (zx,zy,zw,zh), 2)
    surf.blit(font.render("ZONE", True, (140,20,20)), (zx+4, zy+4))

    bp = px(env.bg_pos)
    ip = px(env.int_pos)
    pygame.draw.line(surf, (200,120,120), ip, px(env.zone_center), 1)

    pygame.draw.circle(surf, (100,160,255), bp, int(env.intercept_radius*PX), 1)

    pygame.draw.circle(surf, (20,90,210),  bp, 14)
    pygame.draw.circle(surf, (210,50,20),  ip, 10)
    surf.blit(font.render("B", True, (255,255,255)), (bp[0]-5, bp[1]-7))
    surf.blit(font.render("I", True, (255,255,255)), (ip[0]-4, ip[1]-7))
    hud = [
        f"Ep {ep+1}  Step {step:3d}",
        f"Blocking : {env._block():.2f}",
        f"BG-INT d : {env._dist(env.bg_pos, env.int_pos):.2f}",
        f"INT-zone : {env._dist(env.int_pos, env.zone_center):.2f}",
        f"INT speed: {env.int_max_speed:.2f}",
    ]
    for i, line in enumerate(hud):
        surf.blit(font.render(line, True, (40,40,40)), (6, 6+i*16))
    if banner:
        col = (20,160,60) if "WIN" in banner or "INTERCEPT" in banner else (200,30,30)
        b   = font2.render(banner, True, col)
        surf.blit(b, (PX//2 - b.get_width()//2, PX//2 - 12))

    return np.transpose(pygame.surfarray.pixels3d(surf), (1,0,2))


all_frames = []

for ep in range(EPISODES):
    env = AdversarialEnv(intruder_speed=0.05)
    obs, _ = env.reset()
    done = False
    step = 0
    ep_frames = []

    while not done:
        ba, _ = bg.predict(obs,  deterministic=True)
        ia, _ = itr.predict(obs, deterministic=True)
        obs, _, _, term, trunc, info = env.step(np.concatenate([ba, ia]))
        step += 1
        done = term or trunc
        ep_frames.append(draw_frame(env, step, ep))

    if info['intercepted']:
        banner = "INTERCEPTED!"
    elif info['success']:
        banner = "BG WIN (timeout)"
    else:
        banner = "ZONE BREACHED!"
    freeze = draw_frame(env, step, ep, banner)
    ep_frames.extend([freeze] * int(FPS * 1.5))

    ep_frames.extend([np.zeros((PX, PX, 3), dtype=np.uint8)] * int(FPS * 0.4))

    all_frames.extend(ep_frames)
    print(f"Ep {ep+1}: {banner} in {step} steps ({len(ep_frames)} frames)")

print(f"\nWriting {len(all_frames)} frames → {GIF_PATH}  (~30s)...")
os.makedirs(os.path.dirname(GIF_PATH), exist_ok=True)
imageio.mimsave(GIF_PATH, all_frames, fps=FPS, loop=0)
size_mb = os.path.getsize(GIF_PATH) / 1024 / 1024
print(f"✅ Done — {size_mb:.1f} MB")
display(Image(filename=GIF_PATH))

FileNotFoundError: [Errno 2] No such file or directory: '/content/models/bodyguard_final.zip'